# Question 2: Forward Vol Time Series

Build a panel of forward vols by stripping the cap curve on each date in the time series.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import norm
from scipy.optimize import brentq
from pathlib import Path
from IPython.display import display

%matplotlib inline
plt.rcParams['figure.figsize'] = (14, 5)
pd.set_option('display.max_columns', 15)

DATA_PATH = Path('Data')

In [ ]:
# Load cap vol time series (normal vol, bp)
cap_raw = pd.read_excel(DATA_PATH / 'project_cap_vol_ts.xlsx', sheet_name='cap', index_col=0)
cap_maturities = cap_raw.loc['maturity'].astype(float)
cap_data = cap_raw.drop(index='maturity').astype(float)
cap_data.index = pd.to_datetime(cap_data.index)
cap_data.columns = (4 * cap_maturities.values).round(0) / 4
cap_data.columns.name = 'maturity'
cap_data = cap_data.T.drop_duplicates().T

# Load SOFR swap rates (annually compounded, decimal)
sofr_raw = pd.read_excel(DATA_PATH / 'project_cap_vol_ts.xlsx', sheet_name='sofr', index_col=0)
sofr_maturities = sofr_raw.loc['maturity'].astype(float)
sofr_data = sofr_raw.drop(index='maturity').astype(float)
sofr_data.index = pd.to_datetime(sofr_data.index)
sofr_data.columns = (12 * sofr_maturities.values).round(0) / 12
sofr_data.columns.name = 'maturity'
sofr_data = sofr_data.T.drop_duplicates().T
sofr_data = sofr_data / 100

# Validation curves
curves_val = pd.read_excel(
    DATA_PATH / 'cap_curves_2025-06-30.xlsx',
    sheet_name='rate curves 2025-06-30'
).set_index('tenor')

print(f'Cap data: {cap_data.shape[0]} days, maturities: {list(cap_data.columns)}')
print(f'SOFR data: {sofr_data.shape[0]} days, tenors: {sofr_data.shape[1]}')

## Stripping Pipeline

For each date: interpolate to quarterly grid, build discount/forward curves, convert normal vol to Black vol, bootstrap forward vols.

In [ ]:
def blacks_formula(T, vol, strike, fwd, discount=1.0, is_call=True):
    """Black's formula for a caplet. T = option expiry (fixing date)."""
    if vol <= 0 or T <= 0:
        intrinsic = max(fwd - strike, 0) if is_call else max(strike - fwd, 0)
        return discount * intrinsic
    sig_sqrt_T = vol * np.sqrt(T)
    d1 = (np.log(fwd / strike) + 0.5 * vol**2 * T) / sig_sqrt_T
    d2 = d1 - sig_sqrt_T
    if is_call:
        return discount * (fwd * norm.cdf(d1) - strike * norm.cdf(d2))
    else:
        return discount * (strike * norm.cdf(-d2) - fwd * norm.cdf(-d1))


def build_quarterly_curves(date, cap_data, sofr_data):
    """
    Build quarterly rate/vol curves from project time series for a single date.
    Returns DataFrame indexed by quarterly tenors with columns:
    swap_rates, discounts, forwards, flat_vols_black
    """
    swaps = sofr_data.loc[date].dropna()
    cap_vols_bp = cap_data.loc[date].dropna()

    max_tenor = min(cap_vols_bp.index.max(), swaps.index.max())
    tenors = np.arange(0.25, max_tenor + 0.01, 0.25)
    tenors = np.round(tenors, 2)

    # Interpolate swap rates to quarterly grid
    swap_q = pd.Series(
        np.interp(tenors, swaps.index.values, swaps.values),
        index=tenors, name='swap rates'
    )

    # Discount factors (annually compounded)
    discounts = pd.Series((1 + swap_q.values) ** (-tenors), index=tenors, name='discounts')

    # Simple quarterly forward rates: f(t) = (Z(t-dt)/Z(t) - 1) / dt
    forwards = pd.Series(np.nan, index=tenors, name='forwards')
    for i in range(1, len(tenors)):
        forwards.iloc[i] = (discounts.iloc[i - 1] / discounts.iloc[i] - 1) / 0.25

    # Interpolate flat vols to quarterly grid (normal bp)
    # Extrapolate: use 1Y vol for tenors < 1Y
    cap_tenors = cap_vols_bp.index.values
    cap_vals = cap_vols_bp.values
    ext_tenors = np.concatenate([[0.25], cap_tenors[cap_tenors >= 1.0]])
    ext_vals = np.concatenate([[cap_vals[0]], cap_vals[cap_tenors >= 1.0]])
    flat_bp_q = np.interp(tenors, ext_tenors, ext_vals)

    # Convert normal bp -> Black vol: sigma_B = (sigma_N / 10000) / swap_rate
    flat_vols_black = pd.Series(flat_bp_q / 10000 / swap_q.values, index=tenors, name='flat vols')
    flat_vols_black[tenors < 0.5] = np.nan

    return pd.DataFrame({
        'swap rates': swap_q,
        'discounts': discounts,
        'forwards': forwards,
        'flat vols': flat_vols_black
    }, index=tenors)


def strip_forward_vols(curves, dt=0.25, notional=100):
    """
    Bootstrap forward vols from flat cap vols.
    Each cap at maturity T uses ATM strike = swap_rate(T).
    Earlier caplets are re-priced with each new cap's strike.
    """
    tenors = curves.index.values
    flat_vols = curves['flat vols'].values
    swaps = curves['swap rates'].values
    fwds = curves['forwards'].values
    discs = curves['discounts'].values
    n = len(tenors)

    fwd_vols = np.full(n, np.nan)

    # First cap at 0.5Y: single caplet, fwd vol = flat vol
    first_idx = np.argmin(np.abs(tenors - 0.5))
    fwd_vols[first_idx] = flat_vols[first_idx]

    for i in range(first_idx + 1, n):
        if np.isnan(flat_vols[i]):
            continue

        K = swaps[i]  # ATM strike for cap at this maturity
        sigma_flat = flat_vols[i]

        # Price full cap using flat vol
        cap_price = 0.0
        for j in range(first_idx, i + 1):
            fix_t = tenors[j] - dt
            cap_price += notional * dt * blacks_formula(
                fix_t, sigma_flat, K, fwds[j], discs[j])

        # Re-price earlier caplets with this cap's strike and their forward vols
        known_sum = 0.0
        for j in range(first_idx, i):
            fix_t = tenors[j] - dt
            known_sum += notional * dt * blacks_formula(
                fix_t, fwd_vols[j], K, fwds[j], discs[j])

        last_caplet = cap_price - known_sum

        if last_caplet <= 0:
            fwd_vols[i] = fwd_vols[i - 1]
            continue

        fix_t = tenors[i] - dt
        try:
            fwd_vols[i] = brentq(
                lambda vol: notional * dt * blacks_formula(
                    fix_t, vol, K, fwds[i], discs[i]) - last_caplet,
                1e-6, 10.0)
        except (ValueError, RuntimeError):
            fwd_vols[i] = fwd_vols[i - 1]

    return pd.Series(fwd_vols, index=tenors, name='fwd vols')


def process_date(date, cap_data, sofr_data):
    """Full pipeline for a single date. Returns curves DataFrame with fwd vols."""
    curves = build_quarterly_curves(date, cap_data, sofr_data)
    curves['fwd vols'] = strip_forward_vols(curves)
    return curves

### Pipeline Validation (2025-06-30)

Compare bootstrapped forward vols against the processed validation file.

In [ ]:
# Validate bootstrap logic using the validation file's own data (should match exactly)
fwd_vols_check = strip_forward_vols(curves_val.rename(columns={'swap rates': 'swap rates', 'fwd vols': 'fwd vols_ref'}))
comp_exact = pd.DataFrame({
    'bootstrap': fwd_vols_check,
    'reference': curves_val['fwd vols']
}).dropna()
print(f'Bootstrap validation (using validation file data): max error = {(comp_exact["bootstrap"] - comp_exact["reference"]).abs().max():.2e}')

# Compare pipeline output (project data) vs validation file (different data source)
curves_test = process_date(pd.Timestamp('2025-06-30'), cap_data, sofr_data)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
ax = axes[0]
curves_test['flat vols'].dropna().plot(ax=ax, label='Pipeline (project data)', marker='.', ms=3)
curves_val['flat vols'].dropna().plot(ax=ax, label='Validation file', marker='x', ms=4, linestyle='--')
ax.set_title('Flat Vol (Black)'); ax.set_ylabel('Black vol'); ax.legend(); ax.grid(True, alpha=0.3)

ax = axes[1]
curves_test['fwd vols'].dropna().plot(ax=ax, label='Pipeline (project data)', marker='.', ms=3)
curves_val['fwd vols'].dropna().plot(ax=ax, label='Validation file', marker='x', ms=4, linestyle='--')
ax.set_title('Forward Vol (Black)'); ax.set_ylabel('Black vol'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

print('\nNote: Project data and validation file use different Bloomberg sources,')
print('so flat/fwd vols differ. The bootstrap logic itself is exact (verified above).')

## (a) Weekly Forward Vol Panel

In [ ]:
# Identify common dates and resample to weekly
common_dates = cap_data.dropna(how='any').index.intersection(sofr_data.dropna(how='any').index)
weekly_dates = pd.DatetimeIndex(
    pd.Series(common_dates, index=common_dates).resample('W').last().dropna().values
)
print(f'Processing {len(weekly_dates)} weekly dates: {weekly_dates[0].date()} to {weekly_dates[-1].date()}')

# Key tenors for the panel
KEY_TENORS = [0.5, 1.0, 2.0, 3.0, 5.0]

# Build panel: forward Black vols and normal vols at key tenors
fwd_vol_black = pd.DataFrame(index=weekly_dates, columns=KEY_TENORS, dtype=float)
fwd_vol_normal = pd.DataFrame(index=weekly_dates, columns=KEY_TENORS, dtype=float)

failures = []
for date in weekly_dates:
    try:
        curves = process_date(date, cap_data, sofr_data)
        for tenor in KEY_TENORS:
            if tenor in curves.index:
                fwd_vol_black.loc[date, tenor] = curves.loc[tenor, 'fwd vols']
                # Normal vol = Black vol * forward rate (bp)
                fwd_vol_normal.loc[date, tenor] = (
                    curves.loc[tenor, 'fwd vols'] * curves.loc[tenor, 'forwards'] * 10000
                )
    except Exception as e:
        failures.append((date, str(e)))

print(f'Successfully processed: {fwd_vol_black.dropna(how="all").shape[0]} / {len(weekly_dates)} weeks')
if failures:
    print(f'Failures: {len(failures)}')

fwd_vol_black.columns.name = 'tenor'
fwd_vol_normal.columns.name = 'tenor'

display(fwd_vol_black.dropna().tail())
display(fwd_vol_normal.dropna().tail())

## (b) Forward Vol Dynamics

In [ ]:
REGIMES = [
    (pd.Timestamp('2022-03-01'), pd.Timestamp('2023-07-31'), 'red', 'Hiking'),
    (pd.Timestamp('2023-08-01'), pd.Timestamp('2024-08-31'), 'gray', 'Pause'),
    (pd.Timestamp('2024-09-01'), pd.Timestamp('2025-12-31'), 'green', 'Easing'),
]

fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Black forward vols
ax = axes[0]
for tenor in KEY_TENORS:
    fwd_vol_black[tenor].dropna().plot(ax=ax, label=f'{tenor}Y', linewidth=1)
ax.set_title('Forward Vol by Tenor (Black)')
ax.set_ylabel('Black vol')
ax.legend(); ax.grid(True, alpha=0.3)

# Normal forward vols
ax = axes[1]
for tenor in KEY_TENORS:
    fwd_vol_normal[tenor].dropna().plot(ax=ax, label=f'{tenor}Y', linewidth=1)
ax.set_title('Forward Vol by Tenor (Normal, bp)')
ax.set_ylabel('Normal vol (bp)')
ax.legend(); ax.grid(True, alpha=0.3)

for a in axes:
    for start, end, color, label in REGIMES:
        a.axvspan(start, end, alpha=0.07, color=color)

plt.tight_layout()
plt.show()

In [ ]:
# Term structure slope: 5Y - 0.5Y forward vol spread
fig, ax = plt.subplots(figsize=(14, 4))

spread_black = fwd_vol_black[5.0] - fwd_vol_black[0.5]
spread_black.dropna().plot(ax=ax, color='navy', linewidth=1)
ax.axhline(0, color='black', linewidth=0.5)
ax.set_title('Forward Vol Term Spread (5Y - 0.5Y, Black)')
ax.set_ylabel('Vol spread')
ax.grid(True, alpha=0.3)

for start, end, color, label in REGIMES:
    ax.axvspan(start, end, alpha=0.07, color=color)

plt.tight_layout()
plt.show()

**Discussion (Q2b):**

*TODO: Describe which tenors are most volatile and how forward vol evolved across regimes.*

- **Hiking (2022-23):** Short-tenor forward vols should be elevated and volatile as the market priced rapidly shifting expectations for the terminal rate. The forward vol curve is likely inverted (short > long) or flat.

- **Pause (2023-24):** Short-tenor vol collapses (rates are stable), but longer-tenor vol stays elevated reflecting uncertainty about when cuts begin. The curve should steepen.

- **Easing (2024-25):** Short-tenor vol picks up modestly as rate moves resume. The forward vol curve normalizes toward a typical upward slope.

## (c) Summary Statistics

In [ ]:
# Summary statistics in NORMAL vol (bp) -- appropriate for cross-tenor comparison
print('Forward Vol Summary Statistics (Normal Vol, bp)')
print('=' * 60)
stats_normal = fwd_vol_normal.describe().loc[['mean', 'std', 'min', 'max', 'count']]
display(stats_normal.round(1))

print()
print('Forward Vol Summary Statistics (Black Vol)')
print('=' * 60)
stats_black = fwd_vol_black.describe().loc[['mean', 'std', 'min', 'max', 'count']]
display(stats_black.round(4))

In [ ]:
# Summary statistics by regime
def assign_regime(date):
    if date < pd.Timestamp('2023-08-01'):
        return 'Hiking'
    elif date < pd.Timestamp('2024-09-01'):
        return 'Pause'
    else:
        return 'Easing'

regimes = fwd_vol_normal.index.map(assign_regime)

print('Mean Forward Normal Vol (bp) by Regime')
print('=' * 60)
regime_means = fwd_vol_normal.groupby(regimes).mean()
display(regime_means.round(1))

**Discussion (Q2c):**

*TODO: Interpret the statistics and discuss vol convention choice.*

We report in **normal vol** (bp) for cross-tenor comparisons. Black vol varies mechanically with the rate level: when rates are 5% vs 1%, the same normal vol implies very different Black vols. Normal vol strips out this level effect, giving a cleaner comparison of rate uncertainty across tenors and time periods. This is especially important over 2022-2025 where rates moved from near-zero to 5.3%.

Black vol remains the right input for pricing (Black's formula), but normal vol is the right lens for economic interpretation.